# Coding Assistant on AgentCore: A Walkthrough

In this notebook, you run the Coding Assistant evaluation on Amazon Bedrock AgentCore Runtime, and you learn why each part is there. You use the same tasks, the same hidden tests, and the same judges as in notebooks 01 to 03. You need no Docker and no Harbor. Each trial runs in its own AgentCore session, and a firewall inside the session controls what the agent can reach.

Read the notebook in order. The cells that cost money say so.

### Procedure

1. Learn the approach in three minutes.
2. Check your prerequisites and permissions.
3. Create the resources in your AWS account (bootstrap).
4. Examine a PR review datapoint and a coding datapoint, and the parts that grade them.
5. Build and deploy the task images.
6. Run one PR review trial and one coding trial. Read the results.
7. Run the controls: an empty agent and the merged fix.
8. See what the firewall allows and what it blocks.
9. Run many trials at the same time.
10. Read the results with the noise in mind.
11. **Clean up.** Delete everything that this notebook made.

> **Warning: this notebook creates resources in your AWS account.**
>
> | | |
> |---|---|
> | **What it creates** | 3 IAM roles, 1 ECR repository (about 7 GB of images), 1 S3 bucket, 1 CodeBuild project, 1 CloudWatch log group, and one AgentCore runtime for each task. All have a name that starts with the prefix in section 2 (`evals-nb-` by default) and the tag `Project=evals-streamline-spike`. |
> | **Cost** | About $3 to $5 for the whole notebook, with a 30 % uncertainty. Most of it is the agent model (Codex on Bedrock) and the image builds. While you wait, the images cost about $0.02 per day. A runtime that is not in use costs nothing. |
> | **Time** | About 75 minutes. About 20 of them need you. The rest is waiting for builds, trials, and the deletion of the runtimes. |
> | **When you are done** | **Run section 11.** It deletes everything and shows lists that must be empty. If you stop early, run section 11 anyway: it works from a fresh kernel. |
>
> Use a sandbox account, not a shared production account.

## 1. What and Why: The Approach in Three Minutes

An eval for a coding agent needs three things: a start state that hides the answer, a way to run the agent without letting it find the answer, and a grader that compares its work with a known good result. The module builds each datapoint from your git history in three steps:

1. **Roll back** the repository to the parent commit of the change. The agent sees the code as the reviewer saw it.
2. **Seal** the environment: no git history of the fix, no path to GitHub, and no access to the hidden tests.
3. **Compare to the oracle.** The review that humans gave, or the merged patch, is the gold standard. The grader scores the agent against it.

Notebooks 01 to 03 do this with Docker containers on your computer and Harbor. This notebook does it on AWS. **What stays the same:** the tasks (`datasets/`), the hidden tests, the judges, and the model IDs (`../model_config.py`). **What changes:** where a trial runs and how the seal is made.

| | Docker and Harbor (01 to 03) | AgentCore (this notebook) |
|---|---|---|
| Where a trial runs | A container on your computer | One AgentCore Runtime **session**: a fresh microVM (2 vCPU, 8 GB) for each trial |
| The image | `docker build` on your computer | The task's own Dockerfile plus a small layer, built by AWS CodeBuild into your ECR repository |
| Git seal | The Dockerfile | The same Dockerfile, unchanged |
| Network seal | An allowlist proxy in Docker, or nftables | A firewall **inside the session**: the agent runs in an empty network namespace, and one allowlist proxy is its only way out |
| Verifier seal | The tests are copied in after the agent stops | The tests are in the image, readable only by root, and the agent processes are killed before they are copied out |
| Many trials | A few at the same time, on your laptop | All trials at the same time, one microVM each |

```
your machine (this notebook)          your AWS account, us-east-1
----------------------------          ------------------------------------------------------------------
p2_v2.py bootstrap   ---- API ---->   ECR repository, S3 bucket, CodeBuild project, 3 scoped IAM roles
p2_v2.py build       ---- zip ---->   S3 -> CodeBuild -> ECR image (task Dockerfile + firewall layer)
p2_v2.py deploy      ---- API ---->   one AgentCore runtime for each task
p2_v2.py run         ---- API ---->   one session = one microVM for each trial:

                                      +--------------------------------------------------------------+
                                      | 1. AGENT PHASE   Codex runs as user `agent` in an empty      |
                                      |    network namespace. Its only way out: a relay on localhost |
                                      |    to an allowlist proxy that forwards to the model endpoint |
                                      |    of the agent. Credentials: the agent role (that model).   |
                                      | 2. SWITCH        agent processes are killed, the allowlist   |
                                      |    changes to the judge endpoint, the hidden tests are       |
                                      |    copied in                                                  |
                                      | 3. GRADER PHASE  the module's own tests/test.sh runs.        |
                                      |    Credentials: the grader role (the judge model only).      |
                                      +--------------------------------------------------------------+
                      <---- tar ----  reward.json, agent logs, the patch or the findings
```

`p2_v2.py` is the runner. This notebook calls it as a command, so each step that you run here is a command that you can also run in a terminal. [README.md](README.md) and [TESTER-GUIDE.md](TESTER-GUIDE.md) in this folder have the reference.

> **Note:** This mode is for testers. It runs in `us-east-1` only, with one agent (Codex on Bedrock), and the four tasks of the module.

## 2. Prerequisites

You need Python 3.10 or later, `boto3` 1.44.7 or later (older versions have no AgentCore `platformVersion`), and AWS credentials for a sandbox account in your terminal (`aws sso login`, `AWS_PROFILE`, or access keys). The Region is `us-east-1`. You do not need Docker, Harbor, Node, or the AWS CLI.

This cell sets the settings of the notebook and defines two helpers. `p2()` runs the runner as a child process, prints its output while it runs, and prints a line every 30 seconds when it is quiet (a build has no output for some minutes). The helpers hide 12-digit numbers (account IDs) and your home directory.

`PREFIX` is the name prefix of every resource. A different prefix gives you a second, independent copy. To reuse the images of the README smoke test, set it to `evals-p2b-`.

In [ ]:
import json, os, queue, re, signal, subprocess, sys, threading, time
from pathlib import Path
from IPython.display import Markdown, display

PREFIX = os.environ.get("P2_PREFIX", "evals-nb-")   # every resource name starts with this
AWS_PROFILE = None                                  # for example "my-sandbox". None = the default credential chain
REGION = "us-east-1"

HERE = Path.cwd()
MODULE = HERE.parent
if not (MODULE / "coding_eval.py").exists():
    raise RuntimeError("Open this notebook from the agentcore folder of the Coding Assistant module.")
sys.path.insert(0, str(MODULE))
import coding_eval as ce                            # the helpers of notebooks 02 and 03
from coding_eval import read_json, read_toml

if AWS_PROFILE:
    os.environ["AWS_PROFILE"] = AWS_PROFILE
ENV = {**os.environ, "P2_PREFIX": PREFIX, "PYTHONUNBUFFERED": "1"}

def mask(text):
    return re.sub(r"(?<!\d)\d{12}(?!\d)", "ACCT", text).replace(str(Path.home()), "~")

def table(rows, headers):
    display(Markdown("\n".join(["| " + " | ".join(headers) + " |", "|" + "---|" * len(headers)]
                               + ["| " + " | ".join(str(c) for c in r) + " |" for r in rows])))

def stream(cmd, until=None, beat=30, tick=None, check=True):
    # Run a command, print its output as it comes, return (exit code, lines). `until`: stop printing at the first line that matches.
    t0, lines, quiet = time.time(), [], False
    proc = subprocess.Popen(cmd, cwd=HERE, env=ENV, stdin=subprocess.DEVNULL, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    q = queue.Queue()
    threading.Thread(target=lambda: ([q.put(l) for l in proc.stdout], q.put(None)), daemon=True).start()
    try:
        while True:
            try:
                line = q.get(timeout=beat)
            except queue.Empty:
                print(f"   ... still running, {time.time() - t0:.0f} s" + (tick() if tick else ""), flush=True)
                continue
            if line is None:
                break
            line = mask(line.rstrip())
            lines.append(line)
            quiet = quiet or bool(until and re.search(until, line))
            if not quiet:
                print(line, flush=True)
    except KeyboardInterrupt:                        # ask the runner to stop its sessions, then stop waiting
        proc.send_signal(signal.SIGINT)
        try:
            proc.wait(60)
        except subprocess.TimeoutExpired:
            proc.kill()
        raise
    rc = proc.wait()
    print(f"[exit {rc}, {time.time() - t0:.0f} s]")
    if check and rc != 0:
        raise RuntimeError("The command failed. Read the last lines above.")
    return rc, lines

def p2(*args, **kw):
    return stream([sys.executable, "-u", "p2_v2.py", "--prefix", PREFIX, *args], **kw)

print(f"prefix {PREFIX}, module {MODULE.name}, working folder {HERE.name}")

### Check Python, boto3, and the AWS identity

The cell shows the name of your role or user and hides the account ID. If a check fails, it tells you what to do.

In [ ]:
import importlib.metadata as md
import boto3

def version(v):
    return tuple(int(x) for x in re.findall(r"\d+", v)[:3])

print(f"{'OK ' if sys.version_info >= (3, 10) else 'ERR'} Python {sys.version.split()[0]} (need 3.10 or later)")
ok_boto = version(md.version("boto3")) >= (1, 44, 7)
print(f"{'OK ' if ok_boto else 'ERR'} boto3 {md.version('boto3')} (need 1.44.7 or later)")
if not ok_boto:
    raise RuntimeError("Run `%pip install -U 'boto3>=1.44.7'` in a cell, restart the kernel, and run this cell again.")

session = boto3.Session()
try:
    who = session.client("sts", region_name=REGION).get_caller_identity()
except Exception as e:
    raise RuntimeError(f"No usable AWS credentials ({type(e).__name__}). Run `aws sso login` or set AWS_PROFILE in the settings cell.") from None

kind, _, rest = who["Arn"].split(":", 5)[5].partition("/")
name = rest.split("/")[0] if kind == "assumed-role" else rest.split("/")[-1]
print(f"OK  AWS identity: {kind} {name}, account ************, Region {REGION} (fixed by the runner)")
ACCOUNT_ID, CALLER_ARN, CALLER_NAME, CALLER_KIND = who["Account"], who["Arn"], name, kind   # never printed

### Check your permissions before anything is created

Bootstrap makes IAM roles and many other resources. If your role cannot do that, you want to know now, not after a 6 minute build. The first part of the cell calls one read-only API of each service, which shows a missing permission or a blocking policy. The second part asks the IAM policy simulator about the write actions (it needs `iam:SimulatePrincipalPolicy`; if you do not have it, the cell says so and relies on the first part). The simulator checks your identity policies, but not service control policies, so a pass here is not a promise.

The full list of actions is in section 2 of [TESTER-GUIDE.md](TESTER-GUIDE.md).

In [ ]:
def c(service):
    return session.client(service, region_name=REGION)

reads = {
    "iam": lambda: c("iam").list_roles(MaxItems=1),
    "ecr": lambda: c("ecr").describe_repositories(maxResults=1),
    "s3": lambda: c("s3").list_buckets(),
    "codebuild": lambda: c("codebuild").list_projects(),
    "logs": lambda: c("logs").describe_log_groups(limit=1),
    "bedrock-agentcore-control": lambda: c("bedrock-agentcore-control").list_agent_runtimes(maxResults=1),
    "resourcegroupstaggingapi": lambda: c("resourcegroupstaggingapi").get_resources(ResourcesPerPage=1),
}
problems = []
for service, call in reads.items():
    try:
        call(); print(f"OK  read  {service}")
    except Exception as e:
        problems.append(f"{service}: {type(e).__name__}"); print(f"ERR read  {service}: {type(e).__name__}")

WRITES = ["iam:CreateRole", "iam:PutRolePolicy", "iam:PassRole", "iam:DeleteRole", "sts:AssumeRole", "ecr:CreateRepository",
          "s3:CreateBucket", "s3:PutObject", "codebuild:CreateProject", "codebuild:StartBuild", "logs:CreateLogGroup",
          "bedrock-agentcore:CreateAgentRuntime", "bedrock-agentcore:InvokeAgentRuntimeCommand"]
try:
    arn = CALLER_ARN if CALLER_KIND == "user" else c("iam").get_role(RoleName=CALLER_NAME)["Role"]["Arn"]
    sim = c("iam").simulate_principal_policy(PolicySourceArn=arn, ActionNames=WRITES)["EvaluationResults"]
    denied = [r["EvalActionName"] for r in sim if r["EvalDecision"] != "allowed"]
    print(f"{'OK ' if not denied else 'ERR'} simulator: {len(WRITES) - len(denied)} of {len(WRITES)} write actions allowed")
    problems += [f"not allowed by your policies: {a}" for a in denied]
except Exception as e:
    print(f"--  simulator skipped ({type(e).__name__}); the read checks above are all that this cell can say")

if problems:
    raise RuntimeError("Fix these before you continue (see section 2 of TESTER-GUIDE.md): " + "; ".join(problems))
print("\nPermission pre-check passed.")

## 3. Bootstrap: Create the Resources

`bootstrap` creates what a fresh account lacks. It is idempotent: each step checks if the resource exists, so you can run it again at any time. It takes about 15 seconds.

| Resource | Name | Purpose |
|---|---|---|
| ECR repository | `<prefix>fw` | Holds one image for each task, tagged with the task name |
| S3 bucket | `<prefix>src-<account>-us-east-1` | Holds the build contexts. Private, and objects expire after 3 days |
| Log group | `/aws/codebuild/<prefix>build` | The build logs, kept for 3 days |
| IAM role | `<prefix>codebuild-role` | Lets CodeBuild read the context and push to the repository |
| IAM role | `<prefix>grader-role` | The AgentCore execution role: the credentials of the **grader phase** (the judge model only) |
| IAM role | `<prefix>agent-role` | The credentials of the **agent phase** (the Codex model only). The runner assumes it and hands the temporary credentials to the agent |
| CodeBuild project | `<prefix>build` | Builds the arm64 images |

**Tags and prefix.** Each resource has the tags `Project=evals-streamline-spike` and `Expiry=<date two weeks out>`. Nothing deletes a resource on that date: the tag only tells you, and your colleagues, that it is a temporary test. The name prefix is what the cleanup uses. It never touches a name that does not start with your prefix.

In [ ]:
rc, out = p2("bootstrap", until=r"^\{$")   # the runner prints its report as JSON at the end: the next cell shows it as a table

In [ ]:
report = read_json(HERE / "jobs" / "bootstrap-report.json")
table([(s["step"], s["action"], s["secs"]) for s in report["steps"]], ["step", "result", "seconds"])
print(f"bootstrap took {report['secs']} s. IAM changes need some seconds to reach the other services; "
      f"the runner retried {sum(p['retries'] for p in report['propagation'].values())} times to wait for them.")

### Look at the permissions that the roles have

The two roles that matter for the seal are narrow on purpose. This cell prints the actions and resources of each policy that `bootstrap` installed (`p2_v2.py policies` prints the full documents). Notice that the agent role can call the Codex model and nothing else, and that the grader role can call the judge model and nothing else.

In [ ]:
rc, out = p2("policies", until=r"^###")
docs = re.split(r"^### ", "\n".join(out), flags=re.M)[1:]
for doc in docs:
    title, _, body = doc.partition("\n")
    print(f"\n{title}")
    for st in json.loads(body)["Statement"]:
        actions = st["Action"] if isinstance(st["Action"], list) else [st["Action"]]
        res = st["Resource"] if isinstance(st["Resource"], list) else [st["Resource"]]
        print(f"   {st['Effect']}: {', '.join(actions)[:110]}\n      on {', '.join(res)[:110]}")

## 4. Look at Two Datapoints and What Grades Them

The task files are the same as in notebooks 02 and 03. This section shows a PR review datapoint (reject) and a coding datapoint, and the pieces that grade them. The agent sees `instruction.md` and the container. Only the grader sees `tests/` and `solution/`.

In [ ]:
REVIEW_DIR = MODULE / "datasets" / "awslabs-mcp-pr-review" / "tasks"
CODING_DIR = MODULE / "datasets" / "strands-evals-coding" / "tasks"
TASK_DIRS = {"pr4121": REVIEW_DIR / "awslabs__mcp__pr4121-approve", "pr4615": REVIEW_DIR / "awslabs__mcp__pr4615-reject",
             "pr371": CODING_DIR / "strands-agents__evals__pr371-fix", "pr338": CODING_DIR / "strands-agents__evals__pr338-fix"}
REVIEW_TASK, CODING_TASK = TASK_DIRS["pr4615"], TASK_DIRS["pr371"]

print("PR review datapoint (reject): the files")
ce.show_tree(REVIEW_TASK)

### The PR review datapoint

The agent gets a change that is already applied to the repository, and it must report the defects. PR [#4615](https://github.com/awslabs/mcp/pull/4615) of awslabs/mcp tries to close a bypass of a read-only guard for MySQL. A human reviewer found that the bypass is still open. The oracle (`tests/oracle.json`) lists what the reviewer found. Only a `high` finding blocks a merge. The instruction is the same for each review task, and it does not name the PR.

In [ ]:
oracle = read_json(REVIEW_TASK / "tests" / "oracle.json")
print("expected verdict:", oracle["expected_verdict"], "\n")
for f in oracle["findings"]:
    print(f"  [{f['severity']:<6}] {f['statement'][:105]}...")
print("\n--- what the agent reads (instruction.md, first 12 lines) ---")
print("\n".join((REVIEW_TASK / "instruction.md").read_text().splitlines()[:12]))

### The coding datapoint

The agent gets an issue report and the repository at the commit before the fix. It must write the fix. The issue does not name the file, the function, or the cause. `task.toml` records the cause and the merged fix. The agent never sees `task.toml`.

In [ ]:
meta = read_toml(CODING_TASK / "task.toml")["metadata"]
origin, verifier, story = meta["origin"], meta["verifier"], meta["narrative"]
print(f"origin:  {origin['repo']} PR {origin['pr_numbers']}, issue {origin['issue_numbers']}, base commit {origin['base_commit'][:9]}")
print(f"tests:   {verifier['fail_to_pass']} fail-to-pass and {verifier['pass_to_pass']} pass-to-pass hidden tests")
print("cause:  ", " ".join(story["description"].split()))
print("fix:    ", " ".join(story["solution"].split()))
print("\n--- what the agent reads (instruction.md, first 14 lines) ---")
print("\n".join((CODING_TASK / "instruction.md").read_text().splitlines()[:14]))

### The grader pieces

**PR review.** `tests/score.py` compares the agent's findings with the oracle. First it compares the file and the line. Then an LLM judge (`JUDGE_MODEL_ID`) decides if two findings describe the same problem, with 3 votes and a majority. The signals in `reward.json` are `coverage_required` (the share of `high` findings that the agent found), `coverage_all`, and the precision values. The verdict comes from the findings: `block` if one has the severity `high`. A review **passes** if the verdict is right and, on a reject task, `coverage_required` is 1.0.

**Coding.** `tests/test.sh` runs six stages after the agent stops. The model of the LLM stages is `VERIFIER_MODEL_ID`.

| Stage | What it checks | Files in `verifier/` |
|---|---|---|
| 0 | Save the agent's change as a patch | `agent.patch` |
| 1 | Hidden fail-to-pass and pass-to-pass tests, and the existing test suite. The agent's changes to test files are undone first | `junit.xml`, `pytest.log` |
| 2 | Did the agent write a test that fails on the base and passes on its fix? (reported only) | `agent_tests.json` |
| 3 | A validation agent writes tests for the shape of the agent's change, and a judge reviews them | `validation.json` |
| 4 | Task rubric judge, taste judge (yes/no checks against the merged fix), patch bloat | `rubric.json`, `taste.json`, `bloat.json` |
| 5 | Combine the signals | `reward.json`, `reward_details.json` |

`correct` is 1 when the tests and the validation stories pass. The reward is 1 (a **tasteful solve**) only if, in addition, the rubric score is above 0.5, at least 3 of 5 taste checks pass in each group, and the patch is less than 2 times the size of the merged fix.

In [ ]:
print("tests/ of the coding task:")
for p in sorted((CODING_TASK / "tests").rglob("*")):
    if p.is_file() and "__pycache__" not in p.parts:
        print("  ", p.relative_to(CODING_TASK / "tests"))
print("\n--- the merged fix, the oracle of this task (solution/oracle.patch) ---")
print((CODING_TASK / "solution" / "oracle.patch").read_text()[:1800])

### Where the seal is in this mode

| Path to the answer | Control in AgentCore mode | Where you can see it |
|---|---|---|
| **Git:** `git log --all` shows the fix | The task's own Dockerfile clones one commit and removes remotes and the reflog. The runner uses it unchanged | `environment/Dockerfile` of each task. Section 5 builds it |
| **Network:** the agent looks up the fix on GitHub | The empty network namespace and the allowlist proxy | Section 8 |
| **Verifier:** the agent reads or changes the hidden tests | The tests are in the image as root-only files. They are copied to `/tests` only after the agent's processes are killed | `result.json` of a trial, `phase_switch` |

## 5. Build and Deploy the Images

Each task needs an image and a runtime. **Build** sends the task's Dockerfile and the firewall layer to CodeBuild, which builds an arm64 image and pushes it to your ECR repository. **Deploy** creates an AgentCore runtime that starts sessions from that image.

The build is the slow step. The Dockerfile clones the repository, installs the dependencies, and runs the full test suite of the repository, so a build takes **2 to 6 minutes** (`pr4121` and `pr4615` about 2 minutes, `pr371` about 6 minutes). The images build at the same time, so the wait is the longest build. The runner prints `<task> build started` and then nothing until the build ends. The cell prints a line every 30 seconds with the phase of each build.

Both cells skip what exists, so you can run them again after an interruption. Set `REBUILD = True` to build the images again. A build costs about $0.12 for each image.

In [ ]:
TASKS = ["pr4121", "pr4615", "pr371"]     # the tasks of this notebook (the module also has pr338)
REBUILD = False
ecr, cb = session.client("ecr", region_name=REGION), session.client("codebuild", region_name=REGION)
ctl = session.client("bedrock-agentcore-control", region_name=REGION)

def runtime_name(task):
    return re.sub(r"[^A-Za-z0-9_]", "_", PREFIX) + task

def image_exists(task):
    try:
        ecr.describe_images(repositoryName=PREFIX + "fw", imageIds=[{"imageTag": task}]); return True
    except Exception:
        return False

def runtime_status(task):
    for r in ctl.list_agent_runtimes(maxResults=100)["agentRuntimes"]:
        if r["agentRuntimeName"] == runtime_name(task):
            return r["status"]

def state():
    table([(t, "present" if image_exists(t) else "missing", runtime_status(t) or "missing") for t in TASKS], ["task", "image", "runtime"])

state()

In [ ]:
def build_phase():
    try:
        ids = cb.list_builds_for_project(projectName=PREFIX + "build", sortOrder="DESCENDING")["ids"][:len(TASKS)]
        return ", phases: " + ", ".join(b["currentPhase"] for b in cb.batch_get_builds(ids=ids)["builds"] if b["buildStatus"] == "IN_PROGRESS")
    except Exception:
        return ""

todo = [t for t in TASKS if REBUILD or not image_exists(t)]
if todo:
    print("building", todo, "(2 to 6 minutes)")
    rc, out = p2("build", *todo, until=r"^\[$", tick=build_phase)
else:
    print("All images exist. Set REBUILD = True to build them again.")

In [ ]:
todo = [t for t in TASKS if runtime_status(t) != "READY" or REBUILD]
if todo:
    rc, out = p2("deploy", *todo, until=r"^\[$")
state()

## 6. Run One Trial of Each Kind

`run TASK:N` starts N trials of a task, each in its own session. Each trial does these steps, and the runner times each one:

| Step | What happens |
|---|---|
| `session_start` | AgentCore starts the microVM. A read-only probe checks that the firewall is up |
| `agent_prepare` | The runner sets the allowlist to the agent's model endpoint, gets the agent role credentials, and writes the instruction |
| `agent_loop` | Codex works as user `agent` inside the firewall |
| `phase_switch` | Agent processes are killed, the allowlist and the credentials change, the hidden tests are copied in |
| `verifier` | `tests/test.sh` runs the grader |
| `readback` | The logs and the results are copied to `jobs/<job>/<trial>/` on your computer |

Each run writes to a new folder `jobs/nb-<time>-<name>/`, so you can run a cell again and keep the earlier results.

The first cell below defines what the notebook shows for a trial. It does not call AWS.

In [ ]:
STEPS = {"session_start": "microVM starts, firewall probe", "agent_prepare": "allowlist, credentials, instruction",
         "agent_loop": "the agent works", "phase_switch": "kill agent, switch allowlist and role, copy tests",
         "verifier": "the grader runs", "readback": "copy results to your computer", "total": "the whole trial"}

def new_job(name):
    return f"jobs/nb-{time.strftime('%H%M%S')}-{name}"

def trial_dirs(job):
    return sorted(p.parent for p in Path(job).glob("*/result.json"))

def show_timings(trial):
    res = read_json(trial / "result.json")
    table([(k, res["timings"].get(k), STEPS[k]) for k in STEPS if k in res["timings"]], ["step", "seconds", "what"])
    return res

def review_row(trial):
    res = read_json(trial / "result.json")
    expected = read_toml(TASK_DIRS[res["task"]] / "task.toml")["metadata"]["harvest"]["expected_verdict"]
    path = trial / "artifacts" / "findings.json"
    findings = read_json(path) if path.exists() else []
    predicted = "block" if any(f.get("severity") == "high" for f in findings) else "approve"
    reward = res.get("reward")
    ok = bool(reward) and predicted == expected and (expected == "approve" or reward.get("coverage_required") == 1.0)
    return {"task": res["task"], "expected": expected, "predicted": predicted, "pass": ok, "reward": reward,
            "findings": findings, "errors": res["errors"]}

def show_review(trial):
    r = review_row(trial)
    status = "NO REWARD (invalid trial)" if r["reward"] is None else ("PASS" if r["pass"] else "FAIL")
    print(f"{status}: {r['task']}, expected {r['expected']}, the agent predicted {r['predicted']}")
    for k in ("coverage_required", "coverage_all", "uncredited"):
        print(f"   {k} = {(r['reward'] or {}).get(k)}")
    for f in r["findings"]:
        print(f"   [{f.get('severity')}] {str(f.get('path', '')).split('/')[-1]}:{f.get('line')} {str(f.get('statement', ''))[:100]}")
    for e in r["errors"]:
        print("   error:", e[:200])

def show_fix(trial):
    details = read_json(trial / "verifier" / "reward_details.json")
    flat = details.get("flat", {})
    status = "INVALID" if details.get("invalid") else ("TASTEFUL SOLVE" if flat.get("reward") == 1.0 else
                                                      "CORRECT, NOT TASTEFUL" if flat.get("correct") == 1.0 else "FAIL")
    print(f"{status}: {read_json(trial / 'result.json')['task']}")
    for gate, ok in details.get("checks", {}).items():
        print(f"   {'pass' if ok else 'FAIL'}  {gate}")
    print("   " + ", ".join(f"{k}={flat.get(k)}" for k in ("fail_to_pass", "pass_to_pass", "validation_score", "rubric_score",
                                                           "relative_taste", "practice_alignment", "patch_bloat")))
    for line in details.get("failed_tests", [])[:5]:
        print("   failed test:", line)
    for name, check in (details.get("taste", {}).get("dimensions") or {}).items():
        if isinstance(check, dict) and check.get("passed") is False:
            print(f"   taste check failed: {name}: {str(check.get('reason', ''))[:140]}")

def verifier_timeline(trial):
    # the modification time of each verifier file shows when each stage ended (seconds after the first file)
    rows = [l.split(" ", 2) for l in read_json(trial / "result.json").get("verifier_files_mtime", "").splitlines() if l.strip()]
    rows = sorted((float(t), p.rsplit("/", 1)[-1]) for t, _, p in rows)
    table([(f"+{t - rows[0][0]:.0f}", name) for t, name in rows], ["seconds", "file written"]) if rows else None

### One PR review trial (about 1 minute, about $0.13)

The reject task (PR 4615) is the one from section 4. Codex reads the change, runs the tests, tries inputs, and writes `findings.json`.

In [ ]:
job_review = new_job("review")
rc, out = p2("run", "pr4615:1", "--job", job_review)

In [ ]:
trial = trial_dirs(job_review)[0]
show_review(trial)
res = show_timings(trial)
print("\nfirewall probe at the start of the session:", res["start_probe"])

**Read the result.** A pass needs the verdict `block` and `coverage_required` = 1.0. Codex passes this task in only about 12 % of trials (23 of 185 in the measured sample). That is the agent and the judge, not a fault of the environment, so a fail here is normal. Do not report a low pass rate on this task as a bug. Section 10 says how many trials you need before a rate means something.

The next cell shows how to read what the agent did. The folder of the trial has the same files as a Harbor job.

In [ ]:
print("files of the trial:")
for p in sorted(trial.rglob("*")):
    if p.is_file() and "codex-home" not in p.parts:
        print("  ", p.relative_to(trial))
print("\n--- agent/codex.txt, the transcript of the agent (first 30 lines) ---")
print(mask("\n".join(l[:150] for l in (trial / "agent" / "codex.txt").read_text(errors="ignore").splitlines()[:30])))

### One coding trial (about 7 minutes, about $0.40)

The agent writes a fix, then the six stages of the grader run. Most of the time is the agent (3 to 4 minutes) and the grader (1.5 to 2.5 minutes). Look at the timings: the microVM starts in about 1 second and the firewall adds no wait.

In [ ]:
job_coding = new_job("coding")
rc, out = p2("run", "pr371:1", "--job", job_coding)

In [ ]:
trial = trial_dirs(job_coding)[0]
show_fix(trial)
show_timings(trial)
verifier_timeline(trial)

Each gate of the tasteful solve is listed above. A reward of 0 does not tell you which gate failed: read the gates. If the trial is **invalid** (no reward), the judge or the grader had a fault, not the agent. The runner says why and gives the command to run it again.

### Open the patch and the transcript

`verifier/agent.patch` is what the agent changed. Compare it with the merged fix. `agent/codex.txt` shows each command that the agent ran, and `agent/codex-home/sessions/` has the full event log of Codex.

In [ ]:
print("--- verifier/agent.patch (the agent's change) ---")
print((trial / "verifier" / "agent.patch").read_text()[:2200])
print("--- agent/codex.txt, the end of the transcript ---")
print(mask("\n".join(l[:150] for l in (trial / "agent" / "codex.txt").read_text(errors="ignore").splitlines()[-15:])))

## 7. The Controls: Test the Grader Before You Trust the Agent Score

Before you decide that an agent is good or bad, make sure that the grader gives the right score to two known answers:

- **An empty agent** that changes nothing. On a coding task, it must score 0: if an f2p test passes without a fix, it does not test the bug. On a reject task, it must fail: it finds nothing. **On an approve task, it passes**, because "no findings" is the right answer there. This is why the dataset has both kinds: an agent that approves everything passes the approve task, and the reject task catches it.
- **The merged fix** (the oracle) as the agent. On a coding task, it must be a tasteful solve. If it is not, a mechanism of the reward stack is broken.

Both checks use the real grader and cost only the judge calls (about $0.04 for each coding trial). The empty agent takes about 2 minutes, because the grader still runs its stages. For a review task there is no patch to apply, so there is no oracle run here.

In [ ]:
job_empty = new_job("empty")
rc, out = p2("run", "pr4615:1", "pr4121:1", "pr371:1", "--agent", "empty", "--job", job_empty)

In [ ]:
EXPECT = {"pr4615": "FAIL", "pr4121": "PASS", "pr371": "FAIL"}
rows = []
for t in trial_dirs(job_empty):
    res = read_json(t / "result.json")
    if res["task"] == "pr371":
        got = "NO REWARD" if res["reward"] is None else ("PASS" if res["reward"].get("reward") == 1.0 else "FAIL")
        detail = f"correct={(res['reward'] or {}).get('correct')}, fail_to_pass={(res['reward'] or {}).get('fail_to_pass')}"
    else:
        r = review_row(t)
        got = "NO REWARD" if r["reward"] is None else ("PASS" if r["pass"] else "FAIL")
        detail = f"findings={len(r['findings'])}, predicted {r['predicted']}"
    rows.append((res["task"], "empty agent", EXPECT[res["task"]], got, "ok" if got == EXPECT[res["task"]] else "UNEXPECTED", detail))
table(sorted(rows), ["task", "agent", "expected", "got", "check", "detail"])

In [ ]:
oracle_patch = CODING_TASK / "solution" / "oracle.patch"
job_oracle = new_job("oracle")
rc, out = p2("run", "pr371:1", "--agent", f"patch:{oracle_patch}", "--job", job_oracle)
trial = trial_dirs(job_oracle)[0]
show_fix(trial)

**Read the result.** The merged fix should be a tasteful solve. The taste judge is an LLM and it varies (section 10), so a single `CORRECT, NOT TASTEFUL` on the oracle is possible but rare: run the cell again before you decide that something is broken. If the oracle fails on the hidden tests, the environment or the tests are wrong, and no agent score can be trusted.

## 8. The Firewall: What the Agent Can and Cannot Reach

The agent runs untrusted code, and it must not find the answer. A normal firewall (nftables, iptables) does not load in the microVM, because root has no network-admin capability there. The runner builds the seal in a different way (`image/start.sh` and `image/proxy.py` are all of it):

1. The agent runs in an **empty network namespace**. It has only a loopback interface, so it has no route to anything.
2. A **relay** on `127.0.0.1:3128` inside that namespace forwards to a unix socket.
3. An **allowlist proxy** on the other side of the socket accepts `CONNECT host:443` for the hosts in a root-owned file only. The runner rewrites that file at each phase: the agent's model endpoint in the agent phase, the judge endpoint in the grader phase.
4. The credentials of each phase can call only the model of that phase.

Two checks show this, both from inside a real session. Neither one tries to break out: for that, you need an expert review. The authors ran 43 scripted probes and two LLM red-team runs and found no break-out, but that is not a certified boundary. The environment is `PUBLIC` network mode, so outside the firewall the microVM has internet access.

### Check 1: what each role can call

`scripts/scope_test_v2.py` calls three models with the credentials of each phase. The grader role must reach the judge model only. The agent role must not reach `bedrock-runtime` at all. It uses one-token calls. It needs the `pr4121` runtime, so run it after section 5.

In [ ]:
rc, out = stream([sys.executable, "-u", "scripts/scope_test_v2.py", "--prefix", PREFIX])

### Check 2: what the agent can reach, host by host

This probe runs as user `agent` in the agent phase. For each host it makes an HTTPS request through the proxy, as Codex does, and a direct request that skips the proxy. A host that answers with any HTTP status (even `403` from the service for a call without credentials) was **reached**. A `Tunnel connection failed: 403` is the **proxy** saying no. The probe is a few lines of Python that run inside the session.

In [ ]:
PROBE = '''
import json, urllib.request, urllib.error
HOSTS = ["bedrock-mantle.us-east-1.api.aws", "bedrock-runtime.us-east-1.amazonaws.com", "github.com", "pypi.org", "example.com"]
def get(host, proxy):
    opener = urllib.request.build_opener() if proxy else urllib.request.build_opener(urllib.request.ProxyHandler({}))
    try:
        r = opener.open(f"https://{host}/", timeout=8)
        return f"reached, HTTP {r.status}"
    except urllib.error.HTTPError as e:
        return f"reached, HTTP {e.code}"
    except Exception as e:
        text = str(getattr(e, "reason", e))
        return "blocked by the proxy" if "403" in text else "no route (" + text[:40] + ")"
print(json.dumps({h: [get(h, True), get(h, False)] for h in HOSTS}))
'''
DRIVER = '''
import json, sys, p2_v2 as p2
s = p2.Session("pr4121")
s.start("cat /run/fw.status")
s.put("/tmp/probe.py", sys.stdin.read().encode(), "0644", "agent")
out = {}
for phase, host in (("agent", p2.HOST_MANTLE), ("grader", p2.HOST_RUNTIME)):
    p2.set_phase(s, host, {})
    out[phase] = s.agent("python3 /tmp/probe.py", 120)[0].strip().splitlines()[-1]
s.stop()
print("RESULT " + json.dumps(out))
'''
proc = subprocess.run([sys.executable, "-c", DRIVER], input=PROBE, cwd=HERE, env=ENV, capture_output=True, text=True, timeout=600)
line = next((l for l in proc.stdout.splitlines() if l.startswith("RESULT ")), None)
if line is None:
    raise RuntimeError("The probe failed: " + mask(proc.stderr[-600:] or proc.stdout[-600:]))
rows = []
for phase, answers in json.loads(line[7:]).items():
    for host, (via_proxy, direct) in json.loads(answers).items():
        rows.append((phase, host.replace("us-east-1.", "<region>."), via_proxy, direct))
table(rows, ["phase", "host", "through the proxy", "direct, skipping the proxy"])

**Read the table.** In the agent phase, only the model endpoint of the agent is reached. GitHub and PyPI are blocked, so the agent cannot look up the fix and cannot install a package. Skipping the proxy gives no route, because the namespace has only loopback. In the grader phase, the allowlist holds the judge endpoint instead, and the agent's endpoint is closed. The runner reads the grader credentials as root, outside the agent's namespace, and hands them only to the grader. The agent never gets them.

## 9. Parallelism: Many Trials at the Same Time

`run` starts all of its trials at once, each in its own microVM. A batch takes about as long as its slowest trial. In the measured sample, 12 or 24 trials at once took 10 % to 20 % longer than one trial, and 30 at once ran without throttling. Thirty is the largest size that was tested: for more, run waves one after the other (see [TESTER-GUIDE.md](TESTER-GUIDE.md), section 3).

This cell runs 12 PR review trials, 6 of each task (about 2 minutes and about $1.60). The cell after it compares the wall clock of the batch with the time that the same trials would take one after the other.

In [ ]:
N_EACH = 6        # trials of each review task. 12 in total. The tested limit is 30 at once
job_batch = new_job("batch")
rc, batch_out = p2("run", f"pr4121:{N_EACH}", f"pr4615:{N_EACH}", "--job", job_batch)

In [ ]:
results = [read_json(t / "result.json") for t in trial_dirs(job_batch)]
wall = float(next(re.search(r"batch wall clock (\d+) s", l).group(1) for l in batch_out if "batch wall clock" in l))
serial = sum(r["timings"]["total"] for r in results)
totals = sorted(r["timings"]["total"] for r in results)
print(f"{len(results)} trials, batch wall clock {wall:.0f} s")
print(f"the same trials one after the other: about {serial:.0f} s ({serial / wall:.1f} times as long)")
print(f"trial time: shortest {totals[0]:.0f} s, median {totals[len(totals) // 2]:.0f} s, longest {totals[-1]:.0f} s")
print(f"throttled calls: {sum(r['throttles'] for r in results)}, sessions that needed a start retry: {sum('start_attempts' in r for r in results)}")
print(f"trials without a reward: {sum(r['reward'] is None for r in results)}")

The wall clock is longer than the longest trial by the time for the batch to start and for the slowest result to come back. The runner retries the first call of a session up to 3 times on a new session when AgentCore fails to start it (a log line says so). Later calls are never retried, so the agent's work is never repeated. A trial that still has no reward is listed under the table with the reason and the command to run it again.

## 10. Read the Results With the Noise in Mind

An eval score is a measurement with an error. These are the sources of noise in this mode, and their size in the measured samples:

| Source | Size | What to do |
|---|---|---|
| **Few trials** | The reject task passes about 12 % of the time with Codex. With 6 trials, the 95 % interval around an observed rate is very wide | Run 20 or more trials before you call a rate different from 12 %. About 85 trials separate 12 % from 30 % |
| **The taste judge** | For one patch, the score varied from 0.4 to 1.0 (27 repeats, mean 0.80, standard deviation about 0.19) | Do not read a single taste score. Look at the gates over many trials |
| **The taste gate** | About 5 to 7 % of trials with a correct patch get reward 0 from that gate alone, in every mode that was tested. With 3 trials for each task, the chance of at least one is about 20 % | A single reward 0 with `correct` = 1 may be the judge. Read `taste.json` |
| **Lost trials** | About 10 % of trials had no reward in a run of 60 at once, before the runner retried a failed session start. Some were judge faults | They are not agent fails. Run them again |

The cell below pools all review trials that you ran in this notebook and shows the pass rate with a 95 % Wilson interval. With 15 or so trials, the interval is wide, and it should be.

In [ ]:
from math import sqrt
from collections import defaultdict

def wilson(k, n, z=1.96):
    if n == 0:
        return (0.0, 1.0)
    p, d = k / n, 1 + z * z / n
    mid, half = (p + z * z / (2 * n)) / d, z * sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return max(0.0, mid - half), min(1.0, mid + half)

per_task = defaultdict(lambda: [0, 0, 0])      # trials with a reward, passes, trials without a reward
for job in sorted(Path("jobs").glob("nb-*")):
    for t in trial_dirs(job):
        if "empty" in job.name or "oracle" in job.name or read_json(t / "result.json")["task"] not in ("pr4121", "pr4615"):
            continue
        r = review_row(t)
        per_task[r["task"]][2] += r["reward"] is None
        if r["reward"] is not None:
            per_task[r["task"]][0] += 1
            per_task[r["task"]][1] += r["pass"]
rows = []
for task, (n, k, lost) in sorted(per_task.items()):
    lo, hi = wilson(k, n)
    rows.append((task, TASK_DIRS[task].name.split("-")[-1], f"{k} of {n}", f"{100 * lo:.0f} % to {100 * hi:.0f} %", lost))
table(rows, ["task", "kind", "passed", "95 % interval", "no reward"])

**What this tells you.** The approve task passes in nearly every trial (21 of 21 in the first measured sample), and the reject task passes rarely. Together they show that the agent does not approve everything, and does not block everything. A pass rate of the reject task far from 12 % over 20 or more trials is worth a report. Anything smaller is inside the noise.

**What it does not tell you.** The measured samples are from one account, one day, and one agent. They do not show how a different model, or a different task, would do. In the coding sample (60 trials), 50 of 53 graded trials were a tasteful solve. A reward that is almost always 1 cannot show a worse agent: it needs harder tasks.

## 11. Clean Up

> **Warning: run this section when you are done.** It deletes every resource with the prefix `evals-nb-` (or your `PREFIX`) and the tag `Project=evals-streamline-spike` that this notebook created: runtimes, images, the CodeBuild project, the three roles, the S3 bucket, and the log groups. It never touches a name that does not start with your prefix. The cells in this section do not need the earlier cells: you can run them in a fresh kernel.

A runtime takes 3 to 9 minutes to delete. The cell waits for it, then checks every list. If your policy blocks a delete, the cell prints the commands for you to run by hand.

In [ ]:
import json, os, re, subprocess, sys, time
from pathlib import Path
import boto3

PREFIX = os.environ.get("P2_PREFIX", "evals-nb-")        # the same prefix as in section 2
DELETE_LOCAL_JOBS = False                                # True also deletes the trial folders jobs/nb-* on your computer
REGION = "us-east-1"
assert PREFIX.startswith("evals-"), "refusing a prefix that does not start with evals-"
HERE = Path.cwd()
mask = lambda s: re.sub(r"(?<!\d)\d{12}(?!\d)", "ACCT", s).replace(str(Path.home()), "~")
sess = boto3.Session()
cl = lambda n: sess.client(n, region_name=REGION)
UP = re.sub(r"[^A-Za-z0-9_]", "_", PREFIX)               # runtime names use underscores
account = sess.client("sts").get_caller_identity()["Account"]
BUCKET = f"{PREFIX}src-{account}-{REGION}"

In [ ]:
# Step 1: the runner deletes the runtimes, the ECR repository and images, the CodeBuild project, and the three roles
proc = subprocess.run([sys.executable, "-u", "p2_v2.py", "--prefix", PREFIX, "cleanup"], cwd=HERE, capture_output=True, text=True,
                      env={**os.environ, "P2_PREFIX": PREFIX})
print(mask(proc.stdout + proc.stderr).strip())
print("cleanup exit code:", proc.returncode, "(1 means that something is still busy: the next cells wait and run it again)")

In [ ]:
# Step 2: wait for the runtimes to be gone (a delete takes 3 to 9 minutes), and run the runner again if one was busy
def runtimes():
    return [r for r in cl("bedrock-agentcore-control").list_agent_runtimes(maxResults=100)["agentRuntimes"]
            if r["agentRuntimeName"].startswith(UP)]

t0, reran = time.time(), False
while runtimes() and time.time() - t0 < 900:
    now = runtimes()
    print(f"   {time.time() - t0:.0f} s: " + ", ".join(f"{r['agentRuntimeName']} {r['status']}" for r in now), flush=True)
    if proc.returncode and not reran and not any(r["status"] in ("CREATING", "UPDATING") for r in now):
        reran = True                                      # the runtimes that were busy can be deleted now
        proc = subprocess.run([sys.executable, "-u", "p2_v2.py", "--prefix", PREFIX, "cleanup"], cwd=HERE, capture_output=True,
                              text=True, env={**os.environ, "P2_PREFIX": PREFIX})
        print(mask(proc.stdout + proc.stderr).strip())
    time.sleep(30)
print("runtimes left:", [r["agentRuntimeName"] for r in runtimes()] or "none")

In [ ]:
# Step 3: what the runner does not delete: the S3 bucket and the log groups
by_hand = []
def attempt(what, command, fn):
    try:
        fn(); print("deleted", what)
    except Exception as e:
        code = getattr(e, "response", {}).get("Error", {}).get("Code", type(e).__name__)
        if code in ("NoSuchBucket", "ResourceNotFoundException"):
            return
        print(f"could not delete {what}: {code}"); by_hand.append(command)

s3 = sess.resource("s3", region_name=REGION)
if BUCKET in [b["Name"] for b in cl("s3").list_buckets()["Buckets"]]:
    attempt(f"bucket {mask(BUCKET)}", f"aws s3 rm s3://{BUCKET} --recursive && aws s3api delete-bucket --bucket {BUCKET} --region {REGION}",
            lambda: (s3.Bucket(BUCKET).objects.all().delete(), cl("s3").delete_bucket(Bucket=BUCKET)))
logs = cl("logs")
for pre in (f"/aws/codebuild/{PREFIX}", f"/aws/bedrock-agentcore/runtimes/{UP}"):
    for g in logs.describe_log_groups(logGroupNamePrefix=pre)["logGroups"]:
        attempt(f"log group {g['logGroupName']}", f"aws logs delete-log-group --log-group-name '{g['logGroupName']}' --region {REGION}",
                lambda g=g: logs.delete_log_group(logGroupName=g["logGroupName"]))

### Check that nothing is left

Every list must be empty. The last one is a query by tag: it lists every resource with the tag `Project=evals-streamline-spike`, so it also shows the resources of other prefixes (yours or a colleague's). Look for your prefix only. A runtime that was just deleted can show for a minute in the tag query. Run the cell again if it does.

In [ ]:
iam = cl("iam")
left = {
    "runtimes": [r["agentRuntimeName"] for r in runtimes()],
    "iam roles": [r["RoleName"] for pg in iam.get_paginator("list_roles").paginate() for r in pg["Roles"] if r["RoleName"].startswith(PREFIX)],
    "ecr repositories": [r["repositoryName"] for pg in cl("ecr").get_paginator("describe_repositories").paginate()
                         for r in pg["repositories"] if r["repositoryName"].startswith(PREFIX)],
    "codebuild projects": [p for p in cl("codebuild").list_projects()["projects"] if p.startswith(PREFIX)],
    "s3 buckets": [b["Name"] for b in cl("s3").list_buckets()["Buckets"] if b["Name"].startswith(PREFIX)],
    "log groups": [g["logGroupName"] for pre in (f"/aws/codebuild/{PREFIX}", f"/aws/bedrock-agentcore/runtimes/{UP}")
                   for g in logs.describe_log_groups(logGroupNamePrefix=pre)["logGroups"]],
    "tagged resources of this prefix": [a for pg in cl("resourcegroupstaggingapi").get_paginator("get_resources").paginate(
        TagFilters=[{"Key": "Project", "Values": ["evals-streamline-spike"]}]) for m in pg["ResourceTagMappingList"]
        for a in [m["ResourceARN"]] if PREFIX in a or UP in a],
}
for name, items in left.items():
    print(f"{'OK ' if not items else 'LEFT'} {name}: {[mask(i) for i in items] or 'none'}")
if DELETE_LOCAL_JOBS:
    import shutil
    [shutil.rmtree(p) for p in HERE.glob("jobs/nb-*")]
print("\nEverything is deleted." if not any(left.values()) else "\nSomething is left: read the next cell.")

### If something is left: delete it by hand

Do this only if the cell above shows `LEFT`. Most often, a runtime is still in `DELETING` (wait, and run the check cell again), or your policy does not allow `s3:DeleteObject`, `s3:DeleteBucket`, or `logs:DeleteLogGroup`. The cell prints the commands for the S3 bucket and the log groups. It never prints a command that deletes anything that does not start with your prefix. The AWS CLI is needed only for these commands. Replace `ACCT` with your account ID.

If a role, the repository, or the project is left, `python3 p2_v2.py --prefix <your prefix> cleanup` in a terminal prints the reason, and `scripts/teardown.sh <your prefix>` (or with `--print`) does the same checks with the AWS CLI.

In [ ]:
if by_hand:
    print("Run these in a terminal with a role that may delete them:\n")
    print("\n".join(mask(c) for c in by_hand))
else:
    print("No commands to run by hand." if not any(left.values()) else
          "Nothing to print. Wait for the runtimes, then run the check cell again, or use scripts/teardown.sh.")

## Summary

- The approach is the same as in notebooks 01 to 03: roll back, seal, compare to the oracle. In AgentCore mode, the seal is a Dockerfile (git), a firewall inside the session (network), and a root-only copy of the tests (verifier).
- Each trial is one microVM. Many trials run at the same time, and a batch takes about as long as its slowest trial.
- Run the controls first: an empty agent must fail a coding task and a reject task, and the merged fix must be a tasteful solve.
- Read the gates of a reward, not only the number. A trial without a reward is a grader or an infrastructure fault, not an agent fail.
- A rate from a few trials is noise. The reject task needs 20 or more trials, and the taste judge has a standard deviation of about 0.19.
- Always run the cleanup, and check the lists.

**More:** [README.md](README.md) (design and commands), [TESTER-GUIDE.md](TESTER-GUIDE.md) (permissions, cost, limits, feedback template), and the comparison of the three modes in `../docs/lite-modes-comparison.md`.